# Load and process E. coli data

In [1]:
import numpy as np
import pandas as pd
from Bio import AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load and check `.xmfa` data

In [2]:
def read_xmfa_biopython(file_path):
    blocks = list(AlignIO.parse(file_path, "mauve"))
    return blocks

In [3]:
def remove_gap_columns(alignment_block):
    """
    Takes a Biopython MultipleSeqAlignment block and returns a new block
    with any column containing '-', '.', 'N', or 'n' completely removed.
    """
    align_len = alignment_block.get_alignment_length()

    invalid_chars = {'N', 'n', '-', '.'}
    
    valid_columns = [
        i for i in range(align_len) 
        if not any(char in alignment_block[:, i] for char in invalid_chars)
    ]

    cleaned_records = []
    for record in alignment_block:
        clean_seq_string = "".join([record.seq[i] for i in valid_columns])
        new_record = SeqRecord(Seq(clean_seq_string), id=record.id, description="")
        cleaned_records.append(new_record)

    return MultipleSeqAlignment(cleaned_records), valid_columns

In [4]:
xmfa_path = "../data/ecoli/coli27.xmfa"
aligned_blocks = read_xmfa_biopython(xmfa_path)

print(f"Total blocks successfully loaded: {len(aligned_blocks)}")

Total blocks successfully loaded: 765


In [5]:
def divide_evenly(n, parts):
    quotient, remainder = divmod(n, parts)
    
    # Create the list with the distributed remainder
    parts = [quotient + 1] * remainder + [quotient] * (parts - remainder)
    
    return parts

In [8]:
block_info_df = pd.read_csv("../data/ecoli/ecoli_block_info.csv")
block_info_df['Gene_Length_raw'] = None
block_info_df['Start_pos_raw'] = None
block_info_df['End_pos_raw'] = None
block_info_df.head()

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_raw,Start_pos_raw,End_pos_raw
0,0,1200,605790,606989,True,None,None,None
1,1,1724,614003,615726,True,None,None,None
2,2,4601,616186,620786,True,None,None,None
3,3,6950,606990,613939,True,None,None,None
4,4,3797,601974,605770,True,None,None,None


In [9]:
df_index = 0
for i in tqdm(range(len(aligned_blocks)), desc="Processing genes"):
    # get the raw block and remove gap columns
    raw_block = aligned_blocks[i]
    clear_block, clean_to_raw_map = remove_gap_columns(raw_block)

    if clear_block.get_alignment_length() == 0:
        continue

    # add the gene length and start/end positions to the info table
    if clear_block.get_alignment_length() <= 10000:
        start_pos_clean = block_info_df.loc[df_index, 'Start_pos']
        end_pos_clean = block_info_df.loc[df_index, 'End_pos']
        
        block_info_df.loc[df_index, 'Start_pos_raw'] = clean_to_raw_map[0] + start_pos_clean
        block_info_df.loc[df_index, 'End_pos_raw'] = clean_to_raw_map[-1] + start_pos_clean
        block_info_df.loc[df_index, 'Gene_Length_raw'] = block_info_df.loc[df_index, 'End_pos_raw'] - block_info_df.loc[df_index, 'Start_pos_raw'] + 1

        df_index += 1
    else:
        sub_segments = int(clear_block.get_alignment_length() / 10000) + 1
        sub_length = divide_evenly(clear_block.get_alignment_length(), sub_segments)
        sub_start = raw_block[0].annotations.get('start')

        for j in range(sub_segments):
            start_pos_clean = block_info_df.loc[df_index, 'Start_pos']
            end_pos_clean = block_info_df.loc[df_index, 'End_pos']

            block_info_df.loc[df_index, 'Start_pos_raw'] = clean_to_raw_map[start_pos_clean - sub_start] + sub_start
            block_info_df.loc[df_index, 'End_pos_raw'] = clean_to_raw_map[end_pos_clean - sub_start] + sub_start
            block_info_df.loc[df_index, 'Gene_Length_raw'] = block_info_df.loc[df_index, 'End_pos_raw'] - block_info_df.loc[df_index, 'Start_pos_raw'] + 1

            df_index += 1

Processing genes: 100%|██████████| 765/765 [05:04<00:00,  2.52it/s]


In [10]:
block_info_df

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_raw,Start_pos_raw,End_pos_raw
0,0,1200,605790,606989,True,1200,605790,606989
1,1,1724,614003,615726,True,1731,614003,615733
2,2,4601,616186,620786,True,4602,616186,620787
3,3,6950,606990,613939,True,6956,606990,613945
4,4,3797,601974,605770,True,3800,601974,605773
...,...,...,...,...,...,...,...,...
831,761_0,5476,1504811,1510286,True,5481,1504811,1510291
832,761_1,5475,1510287,1515761,True,5496,1510292,1515787
833,762,4170,1541972,1546141,True,4172,1541972,1546143
834,763,6204,1529775,1535978,True,6231,1529775,1536005


-----

In [19]:
print(block_info_df.head())
block_info_df.to_csv("../data/ecoli/ecoli_block_info_updated.csv", index=False)

  Gene_ID  Gene_Length  Start_pos  End_pos  Alignment Gene_Length_raw  \
0       0         1200     605790   606989       True            1200   
1       1         1724     614003   615726       True            1731   
2       2         4601     616186   620786       True            4602   
3       3         6950     606990   613939       True            6956   
4       4         3797     601974   605770       True            3800   

  Start_pos_raw End_pos_raw  
0        605790      606989  
1        614003      615733  
2        616186      620787  
3        606990      613945  
4        601974      605773  
